# GameGuideLM — Custom TinyQwenStudent Study

This notebook implements the additional study requested after the presentation:

- standard reference benchmarks in addition to pass rate/fact coverage;
- explicit answer-validation traces;
- observed prompt/answer maxima;
- separate online RAG inference and offline custom-model training diagrams;
- pretraining, Qwen3-0.6B distillation, grounded adaptation, diversity, and generalization evaluation for the 43.5M model built by the team.

The pipeline is resumable. Generated corpora, teacher data, checkpoints, and report artifacts stay on Google Drive and are not committed to Git.

## 1. Configuration

In [ ]:
from pathlib import Path

GITHUB_USERNAME = "Ez4Allen"
REPO_NAME = "GameGuide-LLM"
BRANCH = "main"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
PROJECT_ROOT = Path("/content/llm_project")
DRIVE_ROOT = Path("/content/drive/MyDrive/LLM_Project")
FRESH_CLONE = False
INSTALL_DEPENDENCIES = True
INSTALL_OPTIONAL_BERTSCORE = False
SMOKE = False
STAGES_TO_RUN = [
    "corpus", "prompts", "teacher", "pretrain", "scratch_distill",
    "pretrain_distill", "grounded_teacher", "game_adapt", "evaluate", "render",
]
print(REPO_URL)
print("Smoke mode:", SMOKE)
print("Stages:", STAGES_TO_RUN)

## 2. Mount Drive, clone/update the repository, and install dependencies

In [ ]:
import os
import shutil
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(DRIVE_ROOT / "huggingface_cache")
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "huggingface_cache/hub")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if FRESH_CLONE and PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT_ROOT)],
        check=True,
    )
else:
    if not (PROJECT_ROOT / ".git").exists():
        raise RuntimeError("/content/llm_project exists but is not a Git repository.")
    dirty = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain"], text=True
    ).strip()
    if dirty:
        raise RuntimeError("Existing clone has uncommitted changes:\n" + dirty)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if INSTALL_DEPENDENCIES:
    requirement = "requirements-metrics.txt" if INSTALL_OPTIONAL_BERTSCORE else "requirements-training.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", requirement], check=True)
print("Git commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run(["nvidia-smi"], check=False)

## 3. Offline correctness gate

In [ ]:
import inspect
import torch
from src.inference.chat_runtime import QwenPairRuntime
from src.inference.speculative import greedy_speculative_decode

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before model stages.")
if "verification_mode" not in inspect.signature(greedy_speculative_decode).parameters:
    raise RuntimeError("The speculative-correctness implementation is missing.")
if "verification_mode" not in inspect.signature(QwenPairRuntime.generate).parameters:
    raise RuntimeError("QwenPairRuntime is missing verification_mode.")
subprocess.run([sys.executable, "-m", "compileall", "-q", "src", "scripts", "tests"], check=True)
print("PASS: imports, compilation, CUDA, and exact/block verification interfaces are ready.")

## 4. Create a persistent run configuration on Drive

In [ ]:
import yaml

SOURCE_CONFIG = PROJECT_ROOT / "configs/custom_model_study.yaml"
RUN_ROOT = DRIVE_ROOT / ("experiments/custom_model_study_smoke" if SMOKE else "experiments/custom_model_study")
RUN_CONFIG = RUN_ROOT / "custom_model_study.yaml"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
config = yaml.safe_load(SOURCE_CONFIG.read_text(encoding="utf-8"))
config["study"]["output_dir"] = str(RUN_ROOT)
qwen06 = DRIVE_ROOT / "models/Qwen3-0.6B"
if (qwen06 / "config.json").exists():
    config["teacher"]["model_name_or_path"] = str(qwen06)
    config["teacher"]["tokenizer_name_or_path"] = str(qwen06)
    config["teacher"]["local_files_only"] = True
if SMOKE:
    config["pretraining"].update({"max_steps": 10, "eval_steps": 5, "save_steps": 10, "eval_batches": 2})
    config["sequence_distillation"].update({"max_steps": 10, "eval_steps": 5, "save_steps": 10, "eval_batches": 2})
    config["game_adaptation"].update({"max_steps": 10, "eval_steps": 5, "save_steps": 10, "eval_batches": 2})
    config["evaluation"].update({"limit": 4, "diversity_prompts": 2, "samples_per_prompt": 2, "max_new_tokens": 16})
RUN_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(RUN_CONFIG)
print(RUN_CONFIG.read_text(encoding="utf-8"))

## 5. Run the resumable custom-model stages

In [ ]:
for stage in STAGES_TO_RUN:
    print("\n" + "=" * 88)
    print("STAGE:", stage)
    print("=" * 88)
    subprocess.run(
        [sys.executable, "scripts/run_custom_model_study.py", "--config", str(RUN_CONFIG), "--stage", stage],
        cwd=PROJECT_ROOT,
        check=True,
    )
print("Completed requested stages.")
print("Study root:", RUN_ROOT)

## 6. Inspect the study summary

Compare `scratch_distill` against `pretrain_distill` for the pretraining effect, and `pretrain_distill` against `game_adapted` for the grounded-adaptation effect. Diversity metrics are mode-collapse diagnostics, not creative-quality objectives.

In [ ]:
import json
import pandas as pd
from IPython.display import display

SUMMARY_PATH = RUN_ROOT / "evaluation/custom_model_study_summary.json"
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
summary_df = pd.DataFrame([
    {"model": model_name, **model_summary["overall"]}
    for model_name, model_summary in summary["summaries"].items()
])
columns = [
    "model", "top1_agreement", "mean_topk_overlap", "mean_js_divergence",
    "mean_entropy_gap", "reference_rouge_l_f1", "reference_chrf",
    "reference_token_f1", "speculative_acceptance_rate",
    "speculative_exact_match", "unique_draft_top1_ratio",
]
display(summary_df[[column for column in columns if column in summary_df.columns]])
display(pd.DataFrame([{"model": name, **metrics} for name, metrics in summary["diversity"].items()]))
print("Report artifacts:", RUN_ROOT / "report")

## 7. Post-process frozen GameGuideLM answers for the professor feedback

In [ ]:
FINAL_QUALITY_ROOT = DRIVE_ROOT / "experiments/gameguidelm_final_evaluation"
QUALITY_ROWS = FINAL_QUALITY_ROOT / "raw/final_quality_rows.jsonl"
FEEDBACK_ROOT = FINAL_QUALITY_ROOT / "professor_feedback"
DETERMINISTIC_ROWS = DRIVE_ROOT / "experiments/gameguidelm_clean_quick/quality/deterministic/rows.jsonl"
command = [
    sys.executable, "scripts/run_professor_feedback_evaluation.py",
    "--quality-rows", str(QUALITY_ROWS),
    "--references", "data/stardew/evaluation/stardew_eval_v1.jsonl",
    "--output-dir", str(FEEDBACK_ROOT),
    "--answer-limit", "192", "--evidence-source-limit", "6",
    "--evidence-character-limit", "14000",
]
if DETERMINISTIC_ROWS.exists():
    command.extend(["--deterministic-rows", str(DETERMINISTIC_ROWS), "--trace-id", "stardew_eval_001"])
if INSTALL_OPTIONAL_BERTSCORE:
    command.append("--bertscore")
subprocess.run(command, cwd=PROJECT_ROOT, check=True)
print("Feedback artifacts:", FEEDBACK_ROOT)

## 8. Freeze metadata for the report

In [ ]:
manifest = {
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "study_config": str(RUN_CONFIG),
    "study_root": str(RUN_ROOT),
    "evaluation_summary": str(SUMMARY_PATH),
    "report_artifacts": str(RUN_ROOT / "report"),
    "claim_boundary": (
        "The custom model uses lightweight project-local pretraining, not foundation-model pretraining. "
        "Formal evaluation prompts are held out from all optimization."
    ),
}
(RUN_ROOT / "COLAB_RUN_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(manifest, indent=2))